## Establecer conexión con SQL

In [1]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
import os


In [2]:
# Definir la ruta de la carpeta destino
output_folder = r"C:\Users\PC\Desktop\ProjecteData\Equip_15\Data"

In [ ]:
try:
    connection = mysql.connector.connect(host='212.227.90.6',
                                         database='Equip_15',
                                         user='Equipo15',
                                         password = 'E1q2u3i4p5o15')
    if connection.is_connected():
        db_Info = connection.get_server_info()
        print("Connected to MySQL Server version ", db_Info)
        RRHH = pd.read_sql(f"SELECT * FROM RRHH_22092025", con=connection)
        
       
except Error as e:
    print("Error while connecting to MySQL", e)

Connected to MySQL Server version  8.0.43-0ubuntu0.24.04.1


C:\Users\xXSrBiscuitXx\AppData\Local\Temp\ipykernel_27476\859305398.py:7: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_Info = connection.get_server_info()
C:\Users\xXSrBiscuitXx\AppData\Local\Temp\ipykernel_27476\859305398.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  RRHH = pd.read_sql(f"SELECT * FROM RRHH_15092025", con=connection)


## Verificar los tipos iniciales

In [3]:
print("Tipos de variables antes de las conversiones:")
print(RRHH.dtypes)
print("\n")

Tipos de variables antes de las conversiones:
ID                          int64
Reason_absence              int64
Month_absence               int64
Day_week                    int64
Seasons                     int64
Transportation_expense      int64
Distance_Residence_Work     int64
Service_time                int64
Age                         int64
Work_load_Average_day      object
Hit_target                  int64
Disciplinary_failure       object
Education                  object
Son                        object
Social_drinker             object
Social_smoker              object
Pet                        object
Weight                      int64
Height                      int64
Body_mass_index             int64
Absenteeism_hours           int64
dtype: object




## Cambiar de numérico a categórico

In [4]:
# ID debe permanecer como int64 ya que es un identificador único
for col in ["Reason_absence", "Month_absence", "Day_week", "Seasons"]:
    RRHH[col] = RRHH[col].astype("object")

## Cambiar de object a numérico

In [5]:
for col in ["Son", "Pet", "Disciplinary_failure", "Social_drinker", "Social_smoker"]:
    # Primero limpiar y luego convertir
    RRHH[col] = RRHH[col].astype(str).str.strip().astype("int64")

## Cambiar separator decimal y tipo a float para Work_load_Average_day

In [6]:
RRHH["Work_load_Average_day"] = (
    RRHH["Work_load_Average_day"]
    .str.replace(",", ".", regex=False) 
    .astype("float64")
)

## Educación - Corrección: Mantener como variable ordinal numérica para análisis de correlación

In [7]:
# Primero crear una columna numérica para análisis
RRHH["Education_numeric"] = RRHH["Education"].astype(str).str.strip().astype("int64")

# Luego crear la columna categórica para visualización
education_map = {
    "1": "High school",
    "2": "Graduate", 
    "3": "Postgraduate",
    "4": "Master/Doctor"
}
RRHH["Education"] = RRHH["Education"].astype(str).str.strip().replace(education_map)

## Eliminar duplicados exactos

In [8]:
RRHH = RRHH.drop_duplicates()
RRHH.shape[0]

706

## MESES - Crear columna de orden y columna con nombre

In [9]:
RRHH["Month_absence_order"] = RRHH["Month_absence"].astype("int64")

month_map = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}
RRHH["Month_absence"] = RRHH["Month_absence"].astype("object").replace(month_map)

## DÍAS DE LA SEMANA - Crear columna de orden y columna con nombre

In [10]:
RRHH["Day_week_order"] = RRHH["Day_week"].astype("int64")

day_map = {
    2: "Lunes", 3: "Martes", 4: "Miercoles", 5: "Jueves", 6: "Viernes"
}
RRHH["Day_week"] = RRHH["Day_week"].astype("object").replace(day_map)

## ESTACIONES DEL AÑO - Crear columna de orden y columna con nombre

In [11]:
RRHH["Seasons_order"] = RRHH["Seasons"].astype("int64")

spanish_season_map = {
    1: "Invierno", 2: "Otono", 3: "Verano", 4: "Primavera"
}
RRHH["Seasons"] = RRHH["Seasons"].astype("object").replace(spanish_season_map)

## Asegurar que todas las variables numéricas continuas estén como float64

In [12]:
numeric_columns = ["Transportation_expense", "Distance_Residence_Work", "Service_time", 
                  "Age", "Hit_target", "Weight", "Height", "Body_mass_index", 
                  "Absenteeism_hours"]

for col in numeric_columns:
    if col in RRHH.columns:
        RRHH[col] = RRHH[col].astype("float64")

## Verificación final de tipos de variables

In [13]:
print("Tipos de variables después de todas las conversiones:")
print(RRHH.dtypes)
print("\n")

Tipos de variables después de todas las conversiones:
ID                           int64
Reason_absence              object
Month_absence               object
Day_week                    object
Seasons                     object
Transportation_expense     float64
Distance_Residence_Work    float64
Service_time               float64
Age                        float64
Work_load_Average_day      float64
Hit_target                 float64
Disciplinary_failure         int64
Education                   object
Son                          int64
Social_drinker               int64
Social_smoker                int64
Pet                          int64
Weight                     float64
Height                     float64
Body_mass_index            float64
Absenteeism_hours          float64
Education_numeric            int64
Month_absence_order          int64
Day_week_order               int64
Seasons_order                int64
dtype: object




## Información resumida del DataFrame

In [14]:
print("Resumen del DataFrame:")
print(f"Número de filas: {RRHH.shape[0]}")
print(f"Número de columnas: {RRHH.shape[1]}")
print("\nColumnas por tipo de dato:")
print(RRHH.dtypes.value_counts())

Resumen del DataFrame:
Número de filas: 706
Número de columnas: 25

Columnas por tipo de dato:
int64      10
float64    10
object      5
Name: count, dtype: int64


## Exportar a Parquet

In [16]:
try:
    # Asegurar que no hay columnas con tipos problemáticos para Parquet
    # Convertir cualquier object restante a string para evitar problemas
    for col in RRHH.select_dtypes(include=['object']).columns:
        RRHH[col] = RRHH[col].astype("string")
    
    # Construir la ruta completa del archivo Parquet
    parquet_path = os.path.join(output_folder, "RRHH_220925_clean.parquet")
    
    # Exportar a Parquet
    RRHH.to_parquet(parquet_path, index=False, engine='pyarrow')
    print(f"✅ DataFrame exportado exitosamente a {parquet_path}")
    
except Exception as e:
    print(f"❌ Error al exportar a Parquet: {e}")

# Exportar a CSV (siempre se ejecuta, independientemente del resultado de Parquet)
try:
    # Construir la ruta completa del archivo CSV
    csv_path = os.path.join(output_folder, "RRHH_220925_clean.csv")
    
    RRHH.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✅ DataFrame exportado exitosamente a {csv_path}")
except Exception as e:
    print(f"❌ Error al exportar a CSV: {e}")

✅ DataFrame exportado exitosamente a C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\RRHH_220925_clean.parquet
✅ DataFrame exportado exitosamente a C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\RRHH_220925_clean.csv


## Verificación adicional de valores nulos

In [15]:
print("\nValores nulos por columna:")
print(RRHH.isnull().sum())


Valores nulos por columna:
ID                         0
Reason_absence             0
Month_absence              0
Day_week                   0
Seasons                    0
Transportation_expense     0
Distance_Residence_Work    0
Service_time               0
Age                        0
Work_load_Average_day      0
Hit_target                 0
Disciplinary_failure       0
Education                  0
Son                        0
Social_drinker             0
Social_smoker              0
Pet                        0
Weight                     0
Height                     0
Body_mass_index            0
Absenteeism_hours          0
Education_numeric          0
Month_absence_order        0
Day_week_order             0
Seasons_order              0
dtype: int64


# Nuevo Clustering

In [22]:
## Cluster
from sklearn.preprocessing import StandardScaler
from kmodes.kprototypes import KPrototypes

# Agrupar por ID 
df_orig = RRHH.groupby("ID").first()[[
    "Age", "Distance_Residence_Work", "Transportation_expense",
    "Son", "Education", "Social_drinker", "Social_smoker"
]].copy()

# Columnas numéricas y categóricas
num_cols = ["Age", "Distance_Residence_Work", "Transportation_expense", "Son"]
cat_cols = ["Education", "Social_drinker", "Social_smoker"]

# Preparamos un dataframe para K-Prototypes: escalamos SOLO las numéricas
df_kp = df_orig.copy()
scaler = StandardScaler()
df_kp[num_cols] = scaler.fit_transform(df_kp[num_cols])

# Aseguramos que las categóricas sean strings (requerido por k-prototypes)
for c in cat_cols:
    df_kp[c] = df_kp[c].astype(str)
# índices de columnas categóricas para k-prototypes
cat_idx = [df_kp.columns.get_loc(c) for c in cat_cols]

# Ejecutar K-Prototypes (AQUI ESTA NUMERO CLUSTERS)
kproto = KPrototypes(n_clusters=4, random_state=42, init='Cao')
clusters = kproto.fit_predict(df_kp.values, categorical=cat_idx)
df_orig["cluster"] = clusters

# Resumen numérico en unidades originales (medias)
cluster_num_summary = df_orig.groupby("cluster")[num_cols].mean()
print("Resumen numérico (unidades originales):")
print(cluster_num_summary.round(3))

# Resumen categórico: proporciones por cluster (por ejemplo % bebedores/fumadores y distribución de Education)
print("\nProporciones por cluster (categóricas):")
for c in cat_cols:
    prop = df_orig.groupby("cluster")[c].value_counts(normalize=True).unstack(fill_value=0)
    print(f"\n== {c} ==")
    print((prop*100).round(1))   # en porcentaje

# Modo (valor más frecuente) por cluster para las categóricas
mode_cat = df_orig.groupby("cluster")[cat_cols].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
print("\nModo (valor más frecuente) de las categóricas por cluster:")
print(mode_cat)

# Tamaño e IDs por cluster
print("\nTamaño por cluster:")
print(df_orig['cluster'].value_counts().sort_index())
print("\nIDs por cluster:")
for c in sorted(df_orig['cluster'].unique()):
    ids = df_orig[df_orig['cluster'] == c].index.tolist()
    print(f"\nCluster {c} ({len(ids)} empleados):")
    print(ids)


## Centroides y score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np
# Centroides interpretables
def mode_series(s):
    m = s.mode()
    return m.iloc[0] if not m.empty else np.nan

centroids_num = df_orig.groupby("cluster")[num_cols].mean()           # medias en unidades originales
centroids_cat = df_orig.groupby("cluster")[cat_cols].agg(mode_series) # moda de categóricas
centroids_readable = pd.concat([centroids_num, centroids_cat], axis=1)

print("\nCentroides interpretables (numérico = media; categórico = moda):")
print(centroids_readable.round(3))

# Silhouette score - Opción A: SOLO numéricas escaladas
X_num_scaled = scaler.transform(df_orig[num_cols])   # df_orig tiene las num en unidades originales
sil_num = silhouette_score(X_num_scaled, df_orig['cluster'])
print(f"\nSilhouette (solo numéricas escaladas): {sil_num:.4f}")

# Silhouette score - Opción B: numéricas + categóricas (One-Hot)
ct = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ],
    remainder="drop"
)

# Nota: usamos df_orig (unidades originales); ColumnTransformer hará el scaling
X_mixed = ct.fit_transform(df_orig[num_cols + cat_cols])
sil_mixed = silhouette_score(X_mixed, df_orig['cluster'])
print(f"Silhouette (numéricas + one-hot categóricas): {sil_mixed:.4f}")

# Tamaño por cluster y muestra de centroides internos de kproto (opcional)
print("\nTamaño por cluster:")
print(df_orig['cluster'].value_counts().sort_index())

# kproto.cluster_centroids_ existe pero tiene formato interno: 
# numéricas (centroides continuos) y categóricas (modas como índices)
try:
    kp_centroids = kproto.cluster_centroids_
    print("\nCentroides internos reportados por kproto (formato interno):")
    print(kp_centroids)
except Exception:
    pass

print("""
NOTAS:
- Silhouette (A) usa sólo numéricas: rápido, pero ignora info categórica.
- Silhouette (B) usa numéricas + one-hot: más representativo para datos mixtos,
  aunque la dimensionalidad sube y puede afectar el valor.
- Interpretación de Silhouette: >0.5 bueno, 0.2-0.5 moderado, <0.2 débil.
- Los centroides 'readable' muestran la media (num) y la moda (cat) por cluster,
  úsalos para nominar/explicar cada cluster.
""")


Resumen numérico (unidades originales):
            Age  Distance_Residence_Work  Transportation_expense    Son
cluster                                                                
0        34.625                   39.500                 302.625  2.125
1        31.222                   20.556                 214.556  0.222
2        41.667                   36.667                 280.667  0.167
3        44.000                   19.846                 192.769  1.615

Proporciones por cluster (categóricas):

== Education ==
Education  Graduate  High school  Master/Doctor  Postgraduate
cluster                                                      
0              12.5         87.5            0.0           0.0
1              22.2         44.4            0.0          33.3
2               0.0        100.0            0.0           0.0
3               7.7         84.6            7.7           0.0

== Social_drinker ==
Social_drinker     0     1
cluster                   
0               12.5  

## Cerrar conexión

In [18]:
connection.close()
